# OAE Mixing Model
> Andrew McGallian

Surface ocean mixed layer responding to different ocean alkalinity enhancement (OAE) and carbon removal strategies.

## CO2 System Solver (shared functions)

In [ ]:
import matplotlib.pyplot as plt

K1      = 10**(-6.5)
K2      = 10**(-10.8)
K_henry = 10**(1.5) / 1000   # mol/L/atm
K_water = 10**(-14)

def _alpha0(h): return h**2            / (h**2 + K1*h + K1*K2)
def _alpha1(h): return K1*h            / (h**2 + K1*h + K1*K2)
def _alpha2(h): return K1*K2           / (h**2 + K1*h + K1*K2)
def _alk_from_dic(DIC_L, h): return DIC_L*(_alpha1(h) + 2*_alpha2(h)) + K_water/h - h

def solve_co2_system(alk_m3, DIC_m3, iterations=100):
    """Solve CO2 system given alk and DIC in mol/m³. Returns (CO2_m3, CO3_m3, pCO2_ppm, pH)."""
    alk_L, DIC_L = alk_m3/1000, DIC_m3/1000
    pH_lo, pH_hi = 1.0, 14.0
    for _ in range(iterations):
        pH_mid = (pH_lo + pH_hi) / 2
        h = 10**(-pH_mid)
        if _alk_from_dic(DIC_L, h) < alk_L: pH_lo = pH_mid
        else:                                pH_hi = pH_mid
    pH = (pH_lo + pH_hi) / 2
    h  = 10**(-pH)
    CO2_L = DIC_L * _alpha0(h)
    CO3_L = DIC_L * _alpha2(h)
    return CO2_L*1000, CO3_L*1000, (CO2_L/K_henry)*1e6, pH

def co2_from_ppm(pCO2_ppm):
    """Dissolved CO2 concentration (mol/m³) for a given atmospheric pCO2 (ppm)."""
    return K_henry * (pCO2_ppm/1e6) * 1000

# Sanity check
CO2, CO3, pCO2, pH = solve_co2_system(2.0, 2.0)
print(f"pH={pH:.2f}, pCO2={pCO2:.1f} ppm   (expected: pH≈8.59, pCO2≈507 ppm)")

## OAE Mixed Layer Model

In [ ]:
MIXED_LAYER_DEPTH = 100    # m
K_GAS             = 3*365  # m/yr (3 m/day piston velocity)

def oae_mixing_model(alk_add_rate, DIC_add_rate, deepwater_exchange, dt=0.1, duration=20):
    """
    Simulate OAE in a 100-m surface ocean mixed layer.

    alk_add_rate      : alkalinity addition rate (mol/m²/yr)
    DIC_add_rate      : total CO2 addition rate  (mol/m²/yr)
    deepwater_exchange: exchange constant with deep water (m/yr)
    """
    h = MIXED_LAYER_DEPTH
    alk0, DIC0 = 2.0, 2.0                          # mol/m³
    CO2_0, _, pCO2_0, _ = solve_co2_system(alk0, DIC0)
    CO2_atm_eq = CO2_0                             # atmosphere stays fixed

    alk_inv = alk0 * h
    DIC_inv = DIC0 * h

    time_points, DIC_conc, alk_conc = [0.0], [DIC0], [alk0]
    pCO2_ocean, co2_flux_list       = [pCO2_0], [0.0]

    t = 0.0
    for _ in range(int(duration/dt)):
        t += dt
        time_points.append(t)

        alk_c = alk_inv / h
        DIC_c = DIC_inv / h
        CO2_c, _, pCO2_c, _ = solve_co2_system(alk_c, DIC_c)
        pCO2_ocean.append(pCO2_c)

        co2_flux = K_GAS * (CO2_atm_eq - CO2_c)    # mol/m²/yr, + = into ocean
        DIC_inv += co2_flux * dt

        DIC_inv += deepwater_exchange * (DIC0 - DIC_c) * dt
        alk_inv += deepwater_exchange * (alk0 - alk_c) * dt

        DIC_inv += DIC_add_rate * dt
        alk_inv += alk_add_rate * dt

        DIC_conc.append(DIC_inv / h)
        alk_conc.append(alk_inv / h)
        co2_flux_list.append(co2_flux)

    return [time_points, DIC_conc, alk_conc, pCO2_ocean, co2_flux_list]

## Experiments

In [ ]:
# Scenario 1: CaO — only alkalinity, no deep mixing
s1 = oae_mixing_model(alk_add_rate=0.1,  DIC_add_rate=0.0,  deepwater_exchange=0)
# Scenario 2: CaCO3 — alk:DIC = 2:1, no mixing
s2 = oae_mixing_model(alk_add_rate=0.1,  DIC_add_rate=0.05, deepwater_exchange=0)
# Scenario 3: CaO + deepwater mixing
s3 = oae_mixing_model(alk_add_rate=0.1,  DIC_add_rate=0.0,  deepwater_exchange=100)
# Scenario 4: Biological pump — remove DIC, mixing on
s4 = oae_mixing_model(alk_add_rate=0.0,  DIC_add_rate=-0.1, deepwater_exchange=100)
# Scenario 5: Bicarbonate addition — equal alk and DIC, mixing on
s5 = oae_mixing_model(alk_add_rate=0.1,  DIC_add_rate=0.1,  deepwater_exchange=100)

scenarios = [s1, s2, s3, s4, s5]
labels    = ['CaO, no mixing', 'CaCO3, no mixing', 'CaO + mixing',
             'Bio pump + mixing', 'Bicarbonate + mixing']

## Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
ax_dic, ax_alk, ax_pco2, ax_flux = axes.flatten()

for s, label in zip(scenarios, labels):
    t, dic, alk, pco2, flux = s
    ax_dic.plot(t, dic,  label=label)
    ax_alk.plot(t, alk,  label=label)
    ax_pco2.plot(t, pco2, label=label)
    ax_flux.plot(t, flux, label=label)

ax_dic.set(xlabel='Time (yr)',  ylabel='mol/m³',       title='Total CO2 Concentration')
ax_alk.set(xlabel='Time (yr)',  ylabel='mol/m³',       title='Alkalinity Concentration')
ax_pco2.set(xlabel='Time (yr)', ylabel='pCO2 (ppm)',   title='Ocean Equilibrium pCO2')
ax_flux.set(xlabel='Time (yr)', ylabel='mol/m²/yr',    title='CO2 Invasion Flux')

for ax in axes.flatten():
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.suptitle('OAE Mixing Model: Five Scenarios', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()